3. Linear Classifier — NumPy or PyTorch

    Implement a linear classifier: y = Wx.
    Train using L2 loss and gradient descent.
    Options:
        NumPy: write your own forward, backward, and update loop.
        PyTorch: use nn.Linear with optimizer and autograd.

Sources
file:///C:/Users/witch/CSC/4444%20-%20Lao/Python%20Machine%20Learning.pdf

In [1]:
import numpy as np

In [2]:
loaded = np.load("x_short_data.npz")
x_data = loaded["data"]

loaded = np.load("y_short_target.npz")
y_target = loaded["data"]

In [3]:
# ...existing code...

# Load and prepare MNIST data
x_data = np.load("x_short_data.npz")["data"]
y_target = np.load("y_short_target.npz")["data"]

X = np.asarray(x_data, dtype=np.float32).reshape(len(x_data), -1)
y = np.asarray(y_target)

# Support either integer labels or one-hot labels
if y.ndim > 1 and y.shape[1] > 1:
    y = np.argmax(y, axis=1)
else:
    y = y.ravel()

# Scale pixel values to [0, 1] if they are stored as [0, 255]
if X.max() > 1:
    X /= 255.0

# Split into training and test sets
rng = np.random.default_rng(42)
indices = rng.permutation(len(X))
split = int(0.8 * len(X))
train_idx, test_idx = indices[:split], indices[split:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


class LinearClassifier:
    def __init__(self, learning_rate=0.1, epochs=20, batch_size=128):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

    def fit(self, X, y):
        self.classes = np.unique(y)
        y_indices = np.searchsorted(self.classes, y)
        targets = np.eye(len(self.classes), dtype=np.float32)[y_indices]

        rng = np.random.default_rng(42)
        self.weights = rng.normal(0, 0.01, size=(X.shape[1], len(self.classes))).astype(
            np.float32
        )
        self.bias = np.zeros(len(self.classes), dtype=np.float32)

        for epoch in range(self.epochs):
            order = rng.permutation(len(X))

            for start in range(0, len(X), self.batch_size):
                batch = order[start : start + self.batch_size]
                X_batch = X[batch]
                target_batch = targets[batch]

                errors = X_batch @ self.weights + self.bias - target_batch
                self.weights -= self.learning_rate * (
                    2 * X_batch.T @ errors / (len(batch) * len(self.classes))
                )
                self.bias -= self.learning_rate * (
                    2 * errors.mean(axis=0) / len(self.classes)
                )

            loss = np.mean((X @ self.weights + self.bias - targets) ** 2)
            print(f"Epoch {epoch + 1}/{self.epochs} - L2 loss: {loss:.4f}")

        return self

    def predict(self, X):
        scores = X @ self.weights + self.bias
        return self.classes[np.argmax(scores, axis=1)]


model = LinearClassifier()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = np.mean(predictions == y_test) * 100
print(f"Test accuracy: {accuracy:.2f}%")

Epoch 1/20 - L2 loss: 0.0486
Epoch 2/20 - L2 loss: 0.0448
Epoch 3/20 - L2 loss: 0.0431
Epoch 4/20 - L2 loss: 0.0426
Epoch 5/20 - L2 loss: 0.0419
Epoch 6/20 - L2 loss: 0.0413
Epoch 7/20 - L2 loss: 0.0412
Epoch 8/20 - L2 loss: 0.0404
Epoch 9/20 - L2 loss: 0.0400
Epoch 10/20 - L2 loss: 0.0397
Epoch 11/20 - L2 loss: 0.0394
Epoch 12/20 - L2 loss: 0.0393
Epoch 13/20 - L2 loss: 0.0392
Epoch 14/20 - L2 loss: 0.0389
Epoch 15/20 - L2 loss: 0.0390
Epoch 16/20 - L2 loss: 0.0387
Epoch 17/20 - L2 loss: 0.0387
Epoch 18/20 - L2 loss: 0.0387
Epoch 19/20 - L2 loss: 0.0388
Epoch 20/20 - L2 loss: 0.0384
Test accuracy: 86.00%


In [4]:
# class Perceptron(object):
#     """Perceptron classifier.
#     Parameters
#     ------------
#     eta : float
#     Learning rate (between 0.0 and 1.0)
#     n_iter : int
#     Passes over the training dataset.
#     random_state : int
#     Random number generator seed for random weight
#     initialization.
#     Attributes
#     -----------
#     w_ : 1d-array
#     Weights after fitting.
#     errors_ : list
#     Number of misclassifications (updates) in each epoch.
#     """
#     def __init__(self, eta=0.01, n_iter=50, random_state=1):
#         self.eta = eta
#         self.n_iter = n_iter
#         self.random_state = random_state
#     def fit(self, X, y):
#         """Fit training data.
#         Parameters
#         ----------
#         X : {array-like}, shape = [n_examples, n_features]
#         Training vectors, where n_examples is the number of
#         examples and n_features is the number of features.
#         y : array-like, shape = [n_examples]
#         Target values.
#         Returns
#         -------
#         self : object
#         """
#         rgen = np.random.RandomState(self.random_state)
#         self.w_ = rgen.normal(loc=0.0, scale=0.01,
#         size=1 + X.shape[1])
#         self.errors_ = []
#         for _ in range(self.n_iter):
#             errors = 0
#             for xi, target in zip(X, y):
#                 update = self.eta * (target - self.predict(xi))
#                 self.w_[1:] += update * xi
#                 self.w_[0] += update
#                 errors += int(update != 0.0)
#             self.errors_.append(errors)
#         return self
#     def net_input(self, X):
#         """Calculate net input"""
#         return np.dot(X, self.w_[1:]) + self.w_[0]
#     def predict(self, X):
#         """Return class label after unit step"""
#         return np.where(self.net_input(X) >= 0.0, 1, -1)

In [5]:
# v1 = np.array([1, 2, 3])
# v2 = 0.5 * v1
# np.arccos(v1.dot(v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

In [6]:
# import os
# import pandas as pd
# import matplotlib.pyplot as plt
# import numpy as np
# # select setosa and versicolor
# y = df.iloc[0:100, 4].values
# y = np.where(y == 'Iris-setosa', -1, 1)
# # extract sepal length and petal length
# X = df.iloc[0:100, [0, 2]].values
# # plot data
# plt.scatter(X[:50, 0], X[:50, 1], color='red', marker='o', label='setosa')
# plt.scatter(X[50:100, 0], X[50:100, 1], color='blue', marker='x', label='versicolor')

# plt.xlabel('sepal length [cm]')
# plt.ylabel('petal length [cm]')
# plt.legend(loc='upper left')
# plt.show()

# ppn = Perceptron(eta=0.1, n_iter=10)
# ppn.fit(X, y)
# plt.plot(range(1, len(ppn.errors_) + 1), ppn.errors_, marker='o')
# plt.xlabel('Epochs')
# plt.ylabel('Number of updates')
# plt.show()

In [7]:
# from matplotlib.colors import ListedColormap
# def plot_decision_regions(X, y, classifier, resolution=0.02):
#     # setup marker generator and color map
#     markers = ('s', 'x', 'o', '^', 'v')
#     colors = ('red', 'blue', 'lightgreen', 'gray', 'cyan')
#     cmap = ListedColormap(colors[:len(np.unique(y))])
    
#     # plot the decision surface
#     x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
#     x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
#     xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, resolution), np.arange(x2_min, x2_max, resolution))
    
#     Z = classifier.predict(np.array([xx1.ravel(), xx2.ravel()]).T)
#     Z = Z.reshape(xx1.shape)
#     plt.contourf(xx1, xx2, Z, alpha=0.3, cmap=cmap)
#     plt.xlim(xx1.min(), xx1.max())
#     plt.ylim(xx2.min(), xx2.max())
    
#     # plot class examples
#     for idx, cl in enumerate(np.unique(y)):
#         plt.scatter(x=X[y == cl, 0], y=X[y == cl, 1], alpha=0.8, c=colors[idx], marker=markers[idx], label=cl, edgecolor='black')

In [8]:
# plot_decision_regions(X, y, classifier=ppn)
# plt.xlabel('sepal length [cm]')
# plt.ylabel('petal length [cm]')
# plt.legend(loc='upper left')
# plt.show()